## Event Distribution

In [0]:
%sql
SELECT event_type, COUNT(*) AS total_events
FROM clean_100_k
GROUP BY event_type;

event_type,total_events
view,97130
purchase,1655
cart,1215


### observation-
##### Most user activity is concentrated at the browsing stage, with very few users moving to cart or purchase. This suggests low engagement beyond initial product discovery.

##### There is a sharp drop from views to cart, indicating that users are not finding enough value or clarity to take the next step.

##### Interestingly, purchases exceed cart events, which suggests either tracking issues or alternative purchase paths that bypass the cart stage.

### Recommendation-
##### Improve product pages (images, reviews, pricing clarity)
##### Encourage add-to-cart (offers, urgency, recommendations)
##### Fix tracking so user journeys are captured correctly

## USER FUNNEL

In [0]:
%sql
WITH user_funnel AS (
  SELECT 
    user_id,
    MAX(CASE WHEN event_type='view' THEN 1 ELSE 0 END) AS viewed,
    MAX(CASE WHEN event_type='cart' THEN 1 ELSE 0 END) AS carted,
    MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchased
  FROM clean_100_k
  GROUP BY user_id
)

SELECT 
  COUNT(*) AS total_users,
  SUM(viewed) AS viewed_users,
  SUM(carted) AS carted_users,
  SUM(purchased) AS purchased_users
FROM user_funnel;

total_users,viewed_users,carted_users,purchased_users
20384,20383,743,1336


### Observation-
##### Almost all users interact at the browsing stage, but only a small percentage proceed to add items to the cart, indicating a major drop-off early in the user journey.

##### The transition from cart to purchase appears relatively strong, suggesting that users with clear intent are likely to complete the transaction.

##### Interestingly, the number of purchases exceeds cart additions, which points to possible gaps in event tracking or alternative purchase flows.

### Recommendation-
##### Improve product pages (better images, reviews, clearer pricing)
##### Add nudges (discounts, “limited stock”, recommendations)
##### Fix tracking to properly capture the cart step

## CONVERSION RATES

In [0]:
%sql
WITH user_funnel AS (
  SELECT 
    user_id,
    MAX(CASE WHEN event_type='view' THEN 1 ELSE 0 END) AS viewed,
    MAX(CASE WHEN event_type='cart' THEN 1 ELSE 0 END) AS carted,
    MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchased
  FROM clean_100_k
  GROUP BY user_id
)

SELECT 
  round(SUM(carted) * 1.0 / SUM(viewed),2) AS view_to_cart_rate,
  round(SUM(purchased) * 1.0 / SUM(carted),2) AS cart_to_purchase_rate
FROM user_funnel;

view_to_cart_rate,cart_to_purchase_rate
0.04,1.80


### Observation-
##### Only ~4% of users who view products add items to cart, indicating a major drop-off at the product consideration stage.

##### The cart-to-purchase conversion rate exceeds 100%, suggesting inconsistencies in event tracking or the presence of alternative purchase paths that bypass the cart stage.

##### This indicates that while purchase intent exists, the tracked user journey is incomplete and may not fully reflect actual behavior.

### Recommendation
##### Improve product pages (images, reviews, pricing clarity)
##### Add incentives to push users toward cart (discounts, urgency)
##### Fix event tracking to properly capture cart behavior

## DROP-OFF ANALYSIS

In [0]:
%sql
WITH user_funnel AS (
  SELECT 
    user_id,
    MAX(CASE WHEN event_type='view' THEN 1 ELSE 0 END) AS viewed,
    MAX(CASE WHEN event_type='cart' THEN 1 ELSE 0 END) AS carted,
    MAX(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) AS purchased
  FROM clean_100_k
  GROUP BY user_id
)

SELECT 
  SUM(CASE WHEN viewed=1 AND carted=0 THEN 1 END) AS drop_after_view,
  SUM(CASE WHEN carted=1 AND purchased=0 THEN 1 END) AS drop_after_cart
FROM user_funnel;

drop_after_view,drop_after_cart
19640,312


### Observation-
##### A significant majority of users drop off immediately after viewing products, with over 19,000 users failing to progress to the cart stage. This highlights a major bottleneck in the early stage of the user journey.

##### In contrast, very few users drop off after adding items to the cart, indicating that users who demonstrate purchase intent are highly likely to complete the transaction.

##### This suggests that the primary issue lies in converting user interest into intent, rather than in the checkout process itself.

## DAILY ACTIVE USERS

In [0]:
%sql
SELECT 
  DATE(event_time) AS date,
  COUNT(DISTINCT user_id) AS dau
FROM clean_100_k
GROUP BY date
ORDER BY date;

date,dau
2019-10-01,20384


### Observation-
##### The dataset shows user activity for a single day, with over 20,000 active users recorded. While this indicates strong platform engagement, the lack of multi-day data limits the ability to analyze trends over time

### Recommendation-
##### To gain deeper insights into user engagement trends, a longer time range of data should be analyzed to evaluate patterns such as retention, growth, and seasonality.

## TIME TO PURCHASE

In [0]:
%sql
WITH user_times AS (
  SELECT 
    user_id,
    MIN(event_time) AS first_event,
    MIN(CASE WHEN event_type='purchase' THEN event_time END) AS first_purchase
  FROM clean_100_k
  GROUP BY user_id
)

SELECT 
  AVG((UNIX_TIMESTAMP(first_purchase) - UNIX_TIMESTAMP(first_event)) / 3600) AS avg_hours_to_purchase
FROM user_times
WHERE first_purchase IS NOT NULL;

avg_hours_to_purchase
0.15479956753160348


### Observation-
##### Users who complete purchases tend to do so within a short time frame (~9 minutes), indicating a fast decision-making process and high purchase intent among converting users.

### Recommendation
##### Optimize product page for quick decisions (clear price, images, reviews)
##### Use urgency triggers (limited stock, time-based offers)
##### Highlight key information upfront (don’t hide details)

## USER SEGMENTATION

In [0]:
%sql
WITH user_level AS (
  SELECT 
    user_id,
    CASE 
      WHEN SUM(CASE WHEN event_type='purchase' THEN 1 ELSE 0 END) > 0 
      THEN 'Buyer' 
      ELSE 'Non-Buyer' 
    END AS user_type
  FROM clean_100_k
  GROUP BY user_id
)

SELECT 
  user_type,
  COUNT(*) AS total_users
FROM user_level
GROUP BY user_type;

user_type,total_users
Non-Buyer,19048
Buyer,1336


### Observation-
##### A large majority of users (~93%) do not convert into buyers, indicating that while the platform attracts significant traffic, it struggles to convert users into paying customers.
##### The analysis reveals that while user traffic is high, conversion remains low, with the majority of users dropping off early in the journey, highlighting a key opportunity to improve engagement and conversion strategies.

### Recommendation-
##### Target non-buyers with retargeting campaigns (ads, emails, discounts)
##### Improve early-stage experience (product page, recommendations)
##### Use nudges (limited stock, offers) to push decisions

## Engagement Depth

In [0]:
%sql
WITH user_events AS (
  SELECT 
    user_id,
    COUNT(*) AS total_events
  FROM clean_100_k
  GROUP BY user_id
)

SELECT 
  CASE 
    WHEN total_events = 1 THEN '1 event'
    WHEN total_events BETWEEN 2 AND 5 THEN '2-5 events'
    ELSE '6+ events'
  END AS engagement_level,
  COUNT(*) AS users
FROM user_events
GROUP BY engagement_level;

engagement_level,users
2-5 events,8712
6+ events,5215
1 event,6457


### Observation-
##### User engagement varies significantly across the platform. A notable portion of users (~6.4K) interact only once and leave, indicating weak initial engagement or lack of immediate value.

##### The majority of users (~8.7K) fall into the moderate engagement category (2–5 interactions), suggesting that while users show interest, they are not sufficiently motivated to take meaningful actions like adding to cart or purchasing.

##### At the same time, a smaller but important segment (~5.2K users) demonstrates high engagement (6+ interactions), indicating strong interest and potential purchase intent, even though many of them may still not convert.

### Recommendation-
##### To improve overall conversion and engagement, the platform should adopt a segmented strategy:

##### For low-engagement users (1 event): Improve the first interaction experience by enhancing product pages, loading speed, and clarity of value to reduce immediate drop-offs.
##### For moderately engaged users (2–5 events): Introduce nudges such as personalized recommendations, limited-time offers, and reminders to encourage progression toward cart and purchase.
##### For highly engaged users (6+ events): Target them with personalized deals, retargeting campaigns, and tailored messaging, as they represent the highest potential for conversion.

## Event Path Analysis

In [0]:
%sql
SELECT 
  user_id,
  CONCAT_WS(' -> ', COLLECT_LIST(event_type)) AS event_path
FROM clean_100_k
GROUP BY user_id
LIMIT 50;

user_id,event_path
306441847,view -> view
362699320,view -> view -> view -> view -> view -> view -> view
370076704,view
372804920,view -> view
372944259,view -> view
391615981,view
400972610,view -> view -> view -> view -> view
409686912,view -> view
414175028,view -> view -> view -> view
414824833,view


### Observation-
##### User journeys are highly skewed toward repeated browsing behavior. Most users interact only through multiple view events without progressing to cart or purchase, indicating that users are exploring but not committing.

##### In several cases, users perform a large number of views (even 15–20+ interactions) without taking any meaningful action, suggesting hesitation or difficulty in decision-making.

##### Additionally, user paths are not linear—some users move back and forth between actions (e.g., cart → view), and a few even complete purchases without a clear cart step. This indicates inconsistent behavior patterns and possible gaps in event tracking.

##### Overall, while engagement exists, it does not effectively translate into conversion.

### Recommendation-
##### To improve conversion, the platform should focus on reducing friction in the decision-making process and guiding users toward meaningful actions:

##### Enhance product pages with better visuals, reviews, and clear pricing to help users make faster decisions instead of repeatedly browsing.
##### Introduce smart nudges such as “recently viewed items,” personalized recommendations, and limited-time offers to push users toward cart and purchase.
##### Target highly engaged users (those with many views) with tailored incentives, as they show strong interest but need a final push to convert.
##### Improve event tracking and funnel consistency to ensure accurate measurement of user journeys, especially for cart and purchase steps.

# Result-


###### The analysis reveals a significant drop-off at the initial stage of the user journey, where most users leave after viewing products and only a small fraction proceed to add items to the cart. The view-to-cart conversion rate is very low (around 4%), indicating that users are not sufficiently convinced to take the next step after browsing. However, users who do convert demonstrate strong purchase intent, often completing transactions within approximately 9 minutes. The funnel also shows inconsistencies, as purchase events exceed cart events, suggesting potential tracking gaps or alternative purchase paths. Additionally, a large majority of users (~93%) do not convert into buyers, highlighting a clear gap between traffic and actual sales. The data further shows that most users disengage early, with around 19K users dropping off before reaching the cart stage, making it the primary bottleneck. User behavior is also non-linear, with repeated browsing and back-and-forth actions rather than a structured progression through the funnel. Engagement levels vary across users—some interact only once, others explore multiple times without converting, while a smaller segment shows high engagement and intent. Notably, even deep browsing does not guarantee conversion, as many highly engaged users still fail to make a purchase. Overall, the core challenge lies in converting user interest into meaningful action.

## Final Recommendation-

###### To improve overall conversion, the platform should focus on strengthening the early stage of the user journey, where most drop-offs occur. Enhancing product pages with better visuals, detailed descriptions, reviews, and clear pricing can help users make quicker and more confident decisions. Introducing personalized recommendations, recently viewed items, and limited-time offers can nudge users from browsing toward taking action. Since highly engaged users show strong intent but still fail to convert, targeted incentives such as discounts or retargeting campaigns can be effective in capturing this segment. It is also important to reduce decision friction by providing comparison options, trust signals, and clear calls-to-action. Additionally, improving event tracking—especially around cart and purchase behavior—will ensure more accurate measurement of the user journey. Overall, the focus should be on converting user interest into action by guiding users more effectively through the decision-making process.